# TerraClimate climate feature preparation for biodiversity forecasting

This notebook downloads the **TerraClimate** data needed for the biodiversity project,
extracts climate values for the biodiversity observation locations, aggregates monthly
values to annual features, and saves a table that can be joined directly to the
biodiversity data.

## Goal

At the end of this notebook, you will have:

- a **site-year climate feature table**
- a **biodiversity + climate merged table**

## Input expected from Notebook 1

This notebook expects a cleaned biodiversity time-series file in long format, for example:

`land_animals_population_timeseries.csv`

The file should contain at least these columns:

- `Latitude`
- `Longitude`
- `Year`

Optional but helpful columns:

- `ID`
- `Binomial`
- `Common_name`
- `Population`

## Climate variables used in this notebook

For a first model, these TerraClimate variables are a good starting point:

- `ppt` = precipitation
- `tmax` = monthly average maximum temperature
- `tmin` = monthly average minimum temperature
- `pdsi` = Palmer Drought Severity Index

You can add more later.


## 1. Install and import packages

If a package is missing on your machine, uncomment the install cell and run it once.


In [ ]:
# Uncomment if needed
# %pip install pandas numpy xarray netCDF4 requests tqdm

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import requests
from tqdm.auto import tqdm

## 2. Define file paths and settings

Adjust the file names if your local project uses a different folder structure.


In [ ]:
# ---------- project paths ----------
project_dir = Path(".")
input_file = project_dir / "land_animals_population_timeseries.csv"

raw_dir = project_dir / "data" / "terraclimate" / "raw"
processed_dir = project_dir / "data" / "terraclimate" / "processed"

raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

# ---------- TerraClimate settings ----------
# Official TerraClimate page:
# https://www.climatologylab.org/terraclimate.html
#
# This notebook uses direct annual NetCDF downloads following the standard
# TerraClimate filename pattern:
# TerraClimate_<variable>_<year>.nc
#
# If the provider changes the download location in the future, update only BASE_URL.

BASE_URL = "https://www.northwestknowledge.net/metdata/data"

variables = ["ppt", "tmax", "tmin", "pdsi"]

# Set to True if you only want to test the workflow on a small subset of years.
test_mode = False
test_years = [2018, 2019, 2020]

## 3. Load the biodiversity data

This is the cleaned long-format data from the EDA notebook.


In [ ]:
bio_df = pd.read_csv(input_file)

bio_df.head()

In [ ]:
required_columns = ["Latitude", "Longitude", "Year"]
missing_cols = [col for col in required_columns if col not in bio_df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns in biodiversity file: {missing_cols}")

bio_df["Year"] = bio_df["Year"].astype(int)

print("Rows:", len(bio_df))
print("Years:", bio_df["Year"].min(), "-", bio_df["Year"].max())
print("Unique site-year combinations before deduplication:",
      bio_df[["Latitude", "Longitude", "Year"]].drop_duplicates().shape[0])

**Result summary:**  
The biodiversity data has been loaded successfully. The key fields for the climate merge
are latitude, longitude, and year. Because many species observations can share the same
site and year, we will deduplicate these combinations before extracting climate data.


## 4. Create the unique site-year table

Climate features only need to be extracted once per unique combination of:

- latitude
- longitude
- year


In [ ]:
site_year_df = (
    bio_df[["Latitude", "Longitude", "Year"]]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

site_year_df.head()

In [ ]:
if test_mode:
    site_year_df = site_year_df[site_year_df["Year"].isin(test_years)].copy()

years_needed = sorted(site_year_df["Year"].unique().tolist())

print("Unique site-year rows:", len(site_year_df))
print("Years needed:", years_needed[:10], "..." if len(years_needed) > 10 else "")

**Result summary:**  
The climate extraction task has been reduced to unique site-year combinations.
This avoids repeated downloads and repeated extraction work for duplicate locations.


## 5. Download the required TerraClimate NetCDF files

This notebook downloads **only the years that are actually needed** for the biodiversity data.

If a file already exists locally, it will not be downloaded again.


In [ ]:
def download_file(url, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists():
        return "exists"

    response = requests.get(url, stream=True, timeout=120)
    response.raise_for_status()

    total_size = int(response.headers.get("content-length", 0))

    with open(destination, "wb") as f, tqdm(
        total=total_size,
        unit="B",
        unit_scale=True,
        desc=destination.name,
        leave=False
    ) as pbar:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
                pbar.update(len(chunk))

    return "downloaded"


def terraclimate_url(variable, year):
    return f"{BASE_URL}/TerraClimate_{variable}_{year}.nc"


download_log = []

for variable in variables:
    variable_dir = raw_dir / variable

    for year in years_needed:
        url = terraclimate_url(variable, year)
        file_path = variable_dir / f"TerraClimate_{variable}_{year}.nc"

        try:
            status = download_file(url, file_path)
            download_log.append({
                "variable": variable,
                "year": year,
                "status": status,
                "file_path": str(file_path)
            })
        except Exception as e:
            download_log.append({
                "variable": variable,
                "year": year,
                "status": f"failed: {e}",
                "file_path": str(file_path)
            })

download_log_df = pd.DataFrame(download_log)
download_log_df.head()

In [ ]:
download_log_df["status"].value_counts(dropna=False)

**Result summary:**  
The notebook attempted to download one TerraClimate file per variable and year.
If some downloads failed, the most likely causes are a temporary network issue or
a change in the provider's file location. In that case, only the `BASE_URL` usually
needs to be updated.


## 6. Helper functions for reading TerraClimate files

TerraClimate files store one year of monthly data. We extract the nearest grid-cell
value for each biodiversity coordinate.


In [ ]:
def find_main_data_var(ds):
    """Return the main climate variable name from a TerraClimate dataset."""
    candidate_vars = [v for v in ds.data_vars if v.lower() not in {"crs"}]
    if len(candidate_vars) == 1:
        return candidate_vars[0]

    preferred = {"ppt", "tmax", "tmin", "pdsi"}
    for var in candidate_vars:
        if var.lower() in preferred:
            return var

    return candidate_vars[0]


def open_terraclimate_file(file_path):
    ds = xr.open_dataset(file_path, decode_times=True, mask_and_scale=True)
    return ds


def extract_points_from_file(file_path, points_df, variable_name):
    ds = open_terraclimate_file(file_path)
    data_var = find_main_data_var(ds)

    extracted = ds[data_var].sel(
        lat=xr.DataArray(points_df["Latitude"].values, dims="points"),
        lon=xr.DataArray(points_df["Longitude"].values, dims="points"),
        method="nearest"
    )

    result = extracted.to_pandas().T
    result.columns = [f"{variable_name}_month_{i:02d}" for i in range(1, result.shape[1] + 1)]

    out = points_df.reset_index(drop=True).copy()
    out = pd.concat([out, result.reset_index(drop=True)], axis=1)

    ds.close()
    return out

## 7. Extract monthly climate values for each variable and year

This step creates a site-year table with 12 monthly columns per variable.


In [ ]:
climate_monthly_df = site_year_df.copy()

for variable in variables:
    monthly_parts = []

    for year in tqdm(years_needed, desc=f"Extracting {variable}"):
        file_path = raw_dir / variable / f"TerraClimate_{variable}_{year}.nc"

        if not file_path.exists():
            print(f"Missing file: {file_path}")
            continue

        points_this_year = site_year_df[site_year_df["Year"] == year][["Latitude", "Longitude", "Year"]].copy()

        if len(points_this_year) == 0:
            continue

        part = extract_points_from_file(file_path, points_this_year, variable)
        monthly_parts.append(part)

    if len(monthly_parts) == 0:
        print(f"No extracted rows for variable {variable}")
        continue

    variable_monthly_df = pd.concat(monthly_parts, ignore_index=True)

    climate_monthly_df = climate_monthly_df.merge(
        variable_monthly_df,
        on=["Latitude", "Longitude", "Year"],
        how="left"
    )

climate_monthly_df.head()

In [ ]:
print("Monthly climate table shape:", climate_monthly_df.shape)
print("Columns:")
climate_monthly_df.columns.tolist()[:20]

**Result summary:**  
The table now contains monthly climate values for each site-year combination.
This is already enough for some models, but for a first forecasting model it is often
simpler to aggregate these monthly values into annual features.


## 8. Create annual climate features

For the first model, the following annual summaries are useful:

- `ppt_annual_sum`
- `tmax_annual_mean`
- `tmin_annual_mean`
- `tmean_annual_mean`
- `pdsi_annual_mean`
- `pdsi_annual_min`

You can add more later if needed.


In [ ]:
def monthly_cols(prefix):
    return [col for col in climate_monthly_df.columns if col.startswith(f"{prefix}_month_")]


ppt_cols = monthly_cols("ppt")
tmax_cols = monthly_cols("tmax")
tmin_cols = monthly_cols("tmin")
pdsi_cols = monthly_cols("pdsi")

climate_features_df = climate_monthly_df[["Latitude", "Longitude", "Year"]].copy()

if ppt_cols:
    climate_features_df["ppt_annual_sum"] = climate_monthly_df[ppt_cols].sum(axis=1)

if tmax_cols:
    climate_features_df["tmax_annual_mean"] = climate_monthly_df[tmax_cols].mean(axis=1)

if tmin_cols:
    climate_features_df["tmin_annual_mean"] = climate_monthly_df[tmin_cols].mean(axis=1)

if tmax_cols and tmin_cols:
    climate_features_df["tmean_annual_mean"] = (
        climate_features_df["tmax_annual_mean"] + climate_features_df["tmin_annual_mean"]
    ) / 2

if pdsi_cols:
    climate_features_df["pdsi_annual_mean"] = climate_monthly_df[pdsi_cols].mean(axis=1)
    climate_features_df["pdsi_annual_min"] = climate_monthly_df[pdsi_cols].min(axis=1)

climate_features_df.head()

In [ ]:
climate_features_df.describe(include="all")

**Result summary:**  
The monthly TerraClimate data has been converted into annual features that are much easier
to join and use in a first forecasting model. The most important ones here are temperature,
precipitation, and drought-related features.


## 9. Merge the annual climate features back to the biodiversity data

The merge uses the shared keys:

- `Latitude`
- `Longitude`
- `Year`


In [ ]:
bio_with_climate_df = bio_df.merge(
    climate_features_df,
    on=["Latitude", "Longitude", "Year"],
    how="left"
)

bio_with_climate_df.head()

In [ ]:
check_cols = [col for col in ["ppt_annual_sum", "tmax_annual_mean", "tmin_annual_mean", "pdsi_annual_mean"] if col in bio_with_climate_df.columns]

print("Merged rows:", len(bio_with_climate_df))
if check_cols:
    print("Rows with any missing annual climate feature:",
          bio_with_climate_df[check_cols].isna().any(axis=1).sum())

**Result summary:**  
The biodiversity observations now include climate features for the same location and year.
This merged table is ready for the next notebook, where lag features and a first baseline
forecasting model can be built.


## 10. Save the outputs

Two output files are saved:

1. `terraclimate_site_year_features.csv`  
   Contains one row per unique latitude-longitude-year combination.

2. `biodiversity_with_terraclimate_features.csv`  
   Contains the biodiversity data with the climate features already merged in.


In [ ]:
site_year_output = processed_dir / "terraclimate_site_year_features.csv"
merged_output = processed_dir / "biodiversity_with_terraclimate_features.csv"

climate_features_df.to_csv(site_year_output, index=False)
bio_with_climate_df.to_csv(merged_output, index=False)

print("Saved:", site_year_output)
print("Saved:", merged_output)

## 11. Optional checks before modeling

Before moving to model training, it is useful to check:

- how many rows have complete climate data
- whether some regions have more missing climate values than others
- whether the climate variables have extreme values that need inspection


In [ ]:
feature_cols = [
    col for col in climate_features_df.columns
    if col not in ["Latitude", "Longitude", "Year"]
]

missing_summary = bio_with_climate_df[feature_cols].isna().mean().sort_values(ascending=False)
missing_summary

In [ ]:
bio_with_climate_df[feature_cols].hist(figsize=(12, 8), bins=30)
None

**Result summary:**  
These checks help confirm that the climate merge worked as expected. If some features show
unexpected missing values or implausible ranges, this should be reviewed before training the
forecasting model.


## 12. Notes for the next notebook

In the forecasting notebook, a good next sequence would be:

1. load `biodiversity_with_terraclimate_features.csv`
2. sort by population and year
3. create lag features for population
4. optionally create lag features for climate
5. train a first baseline model

Possible first model types:

- Linear Regression
- Random Forest Regressor
- XGBoost (later)

For a beginner-friendly first version, Random Forest is a good baseline.
